# Prédiction de la localisation tumorale — Régression logistique ElasticNet

Classification multi-classes (`cort` / `dipg` / `midl`) à partir des blocs **GE** (15 702 features) et **CGH** (1 229 features), avec **régression logistique multinomiale pénalisée ElasticNet**.

## Différences avec la version précédente

| Point | Avant | Maintenant |
|---|---|---|
| Réduction de dim | PCA classique (10 PCs) | **Sparse PCA** (chargements creux, interprétables) |
| Validation | CV simple (biais optimiste) | **Nested CV** 4-fold × 5 répétitions (estimation non biaisée) |
| Pré-traitement | Scree plot fait sur tout le train avant la grille | **Grilles fixées a priori**, pré-traitement *dans* le Pipeline |
| Fusion tardive | `α` optimisé sur prédictions de modèles entraînés sur tout le train | **StackingClassifier** : méta-modèle entraîné sur prédictions *out-of-fold* |
| Comparabilité | Pipelines hétérogènes selon les modèles | **Même base** : VarianceThreshold → (sélection ou Sparse PCA) → StandardScaler → LogReg ElasticNet |
| Stabilité | Stabilité de SelectKBest seulement | **Stabilité de SelectKBest** + **stabilité des chargements Sparse PCA** |


## 0. Préparation — imports, config, grilles fixées a priori

In [1]:
from __future__ import annotations
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.decomposition import SparsePCA
from sklearn.ensemble import StackingClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
)
from sklearn.model_selection import (
    GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold,
    cross_val_score, permutation_test_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ConvergenceWarning.*")

SEED = 42
LABEL_ORDER = ["cort", "dipg", "midl"]

# ── Validation strategies ────────────────────────────────────────
INNER_CV = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
OUTER_CV = RepeatedStratifiedKFold(n_splits=4, n_repeats=5, random_state=SEED)
PERM_CV  = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)

# ── Hyperparameter grids — FIXÉS A PRIORI ────────────────────────
# Justification : avec n=39, on a au plus 38 composantes principales.
# Les grilles couvrent plusieurs ordres de grandeur sans inspection préalable
# des données (pas de scree plot informant le choix de PCA_GRID).
C_GRID            = [0.01, 0.1, 1.0, 10.0]
L1_GRID           = [0.2, 0.5, 0.8]
GE_K_GRID         = [40, 60, 80, 100, 120]
CGH_K_GRID        = [20, 40, 60]
SPCA_N_COMPONENTS = [5, 10, 15, 20]
SPCA_ALPHA        = [1.0, 2.0]   # contrôle la sparsité des chargements

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.precision", 3)


## 1. Chargement des données

In [2]:
DATA_DIR = next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / "data").is_dir()) / "data"
ANNOT_DIR = next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / "data").is_dir()) / "exports" / "glioma_csv"

def to_numeric_frame(df):
    return df.apply(lambda col: pd.to_numeric(
        col.astype(str).str.replace(",", ".", regex=False), errors="coerce"
    ))

def load_block(block_name, split):
    path = DATA_DIR / f"ge_cgh_locIGR__multiblocks__{block_name}__{split}.csv"
    df = pd.read_csv(path).rename(columns={"row_id": "patient_id"}).set_index("patient_id")
    return to_numeric_frame(df)

def load_targets(split):
    path = DATA_DIR / f"ge_cgh_locIGR__multiblocks__y__{split}.csv"
    y_df = pd.read_csv(path).rename(columns={"row_id": "patient_id"}).set_index("patient_id")
    y = y_df[LABEL_ORDER].idxmax(axis=1)
    y.name = "localisation"
    return y

def fill_missing_from_train(train_df, test_df):
    medians = train_df.median(axis=0)
    return train_df.fillna(medians), test_df.fillna(medians)


X_ge_train  = load_block("GE",  "train")
X_ge_test   = load_block("GE",  "test")
X_cgh_train = load_block("CGH", "train")
X_cgh_test  = load_block("CGH", "test")
y_train     = load_targets("train")
y_test      = load_targets("test")

train_ids = X_ge_train.index.intersection(X_cgh_train.index).intersection(y_train.index)
test_ids  = X_ge_test.index.intersection(X_cgh_test.index).intersection(y_test.index)

X_ge_train  = X_ge_train.loc[train_ids]
X_ge_test   = X_ge_test.loc[test_ids]
X_cgh_train = X_cgh_train.loc[train_ids]
X_cgh_test  = X_cgh_test.loc[test_ids]
y_train     = y_train.loc[train_ids]
y_test      = y_test.loc[test_ids]

X_ge_train,  X_ge_test  = fill_missing_from_train(X_ge_train,  X_ge_test)
X_cgh_train, X_cgh_test = fill_missing_from_train(X_cgh_train, X_cgh_test)

# Bloc concaténé pour fusion précoce / Sparse PCA
X_concat_train = pd.concat([X_ge_train.add_prefix("GE__"),  X_cgh_train.add_prefix("CGH__")], axis=1)
X_concat_test  = pd.concat([X_ge_test.add_prefix("GE__"),   X_cgh_test.add_prefix("CGH__")],  axis=1)

print(f"Train : {len(y_train)} échantillons   |   Test : {len(y_test)} échantillons")
print(f"GE  : {X_ge_train.shape[1]:>5d} features   |   CGH : {X_cgh_train.shape[1]:>5d} features")
display(y_train.value_counts().reindex(LABEL_ORDER).to_frame("train").join(
    y_test.value_counts().reindex(LABEL_ORDER).to_frame("test")))


Train : 39 échantillons   |   Test : 14 échantillons
GE  : 15702 features   |   CGH :  1229 features


,train,test
localisation,,
cort,15,5
dipg,16,6
midl,8,3


## 2. Utilitaires

Toutes les évaluations passent par la même fonction `evaluate_pipeline` qui :
1. Lance une **nested CV** (boucle externe d'évaluation, boucle interne de sélection d'HP via GridSearchCV).
2. Fait un **refit final** sur tout le train avec les meilleurs HP.
3. Évalue sur le **test set** (balanced accuracy, accuracy, macro-F1).
4. Calcule un **bootstrap CI 95%** sur la balanced accuracy test.
5. Lance optionnellement un **test de permutation** (B=500) sur le pipeline complet.

Cela garantit que tous les modèles sont jugés sur la même métrique avec le même protocole de validation.


In [3]:
def prediction_metrics(y_true, y_pred):
    return {
        "accuracy":          accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1":          f1_score(y_true, y_pred, average="macro"),
    }


def bootstrap_ci(y_true, y_pred, metric_fn=balanced_accuracy_score,
                 n_boot=2000, seed=SEED):
    rng = np.random.RandomState(seed)
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true_arr), len(y_true_arr), replace=True)
        scores.append(metric_fn(y_true_arr[idx], y_pred_arr[idx]))
    lo, med, hi = np.percentile(scores, [2.5, 50, 97.5])
    return lo, med, hi


def confusion_table(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    return pd.DataFrame(
        matrix,
        index=[f"true_{l}"  for l in LABEL_ORDER],
        columns=[f"pred_{l}" for l in LABEL_ORDER],
    )


def evaluate_pipeline(name, estimator, param_grid,
                      X_train, y_train, X_test, y_test,
                      run_permutation=True, n_permutations=500,
                      verbose=True):
    """Évalue un pipeline avec nested CV + test set + permutation."""
    inner = GridSearchCV(
        estimator, param_grid,
        cv=INNER_CV, scoring="balanced_accuracy",
        refit=True, n_jobs=-1,
    )

    # 1) Nested CV — estimation honnête de la performance
    if verbose: print(f"[{name}] Nested CV en cours...")
    nested_scores = cross_val_score(
        inner, X_train, y_train,
        cv=OUTER_CV, scoring="balanced_accuracy", n_jobs=-1,
    )

    # 2) Refit final sur tout le train
    inner.fit(X_train, y_train)

    # 3) Test set
    train_pred = inner.predict(X_train)
    test_pred  = inner.predict(X_test)
    train_m = prediction_metrics(y_train, train_pred)
    test_m  = prediction_metrics(y_test,  test_pred)
    lo, med, hi = bootstrap_ci(y_test, test_pred)

    # 4) Permutation test (optionnel)
    perm_obs = np.nan
    pval = np.nan
    if run_permutation:
        if verbose: print(f"[{name}] Test de permutation...")
        perm_obs, _, pval = permutation_test_score(
            inner.best_estimator_, X_train, y_train,
            scoring="balanced_accuracy", cv=PERM_CV,
            n_permutations=n_permutations, random_state=SEED, n_jobs=-1,
        )

    result = pd.Series({
        "model":                   name,
        "nested_cv_mean":          nested_scores.mean(),
        "nested_cv_std":           nested_scores.std(),
        "train_balanced_accuracy": train_m["balanced_accuracy"],
        "test_balanced_accuracy":  test_m["balanced_accuracy"],
        "test_bal_acc_CI_95":      f"[{lo:.3f}, {hi:.3f}]",
        "test_macro_f1":           test_m["macro_f1"],
        "train_test_gap":          train_m["balanced_accuracy"] - test_m["balanced_accuracy"],
        "permutation_score":       perm_obs,
        "p_value":                 pval,
        "best_params":             str(inner.best_params_),
    })
    return result, inner, train_pred, test_pred


## 3. Définition du classifieur de base

Tous les modèles partagent la même brique finale : **régression logistique multinomiale ElasticNet** (L1 + L2) avec `class_weight='balanced'` pour gérer le déséquilibre `midl`.


In [4]:
def make_logreg():
    return LogisticRegression(
        penalty="elasticnet", solver="saga",
        class_weight="balanced",
        max_iter=8000, random_state=SEED,
    )


## 4. Modèle 1 — GE seul (SelectKBest + LogReg)

Pipeline standard : on filtre les features à variance nulle, on garde les `k` meilleurs gènes (ANOVA F-test), on standardise, on entraîne la régression logistique. La grille couvre `k ∈ {40, 60, 80, 100, 120}`.


In [5]:
ge_pipeline = Pipeline([
    ("variance", VarianceThreshold()),
    ("select",   SelectKBest(score_func=f_classif)),
    ("scale",    StandardScaler()),
    ("clf",      make_logreg()),
])
ge_grid = {
    "select__k":       GE_K_GRID,
    "clf__C":          C_GRID,
    "clf__l1_ratio":   L1_GRID,
}

ge_result, ge_search, _, ge_test_pred = evaluate_pipeline(
    "GE seul",
    ge_pipeline, ge_grid,
    X_ge_train, y_train, X_ge_test, y_test,
)
display(ge_result.to_frame().T)
display(confusion_table(y_test, ge_test_pred))


[GE seul] Nested CV en cours...


/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:124

[GE seul] Test de permutation...


/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:124

,model,nested_cv_mean,nested_cv_std,train_balanced_accuracy,test_balanced_accuracy,test_bal_acc_CI_95,test_macro_f1,train_test_gap,permutation_score,p_value,best_params
0,GE seul,0.682,0.132,1.0,0.611,"[0.499, 0.857]",0.571,0.389,0.771,0.002,"{'clf__C': 10.0, 'clf__l1_ratio': 0.2, 'select..."


,pred_cort,pred_dipg,pred_midl
true_cort,5,0,0
true_dipg,0,5,1
true_midl,0,3,0


## 5. Modèle 2 — CGH seul (SelectKBest + LogReg)

In [6]:
cgh_pipeline = Pipeline([
    ("variance", VarianceThreshold()),
    ("select",   SelectKBest(score_func=f_classif)),
    ("scale",    StandardScaler()),
    ("clf",      make_logreg()),
])
cgh_grid = {
    "select__k":       CGH_K_GRID,
    "clf__C":          C_GRID,
    "clf__l1_ratio":   L1_GRID,
}

cgh_result, cgh_search, _, cgh_test_pred = evaluate_pipeline(
    "CGH seul",
    cgh_pipeline, cgh_grid,
    X_cgh_train, y_train, X_cgh_test, y_test,
)
display(cgh_result.to_frame().T)
display(confusion_table(y_test, cgh_test_pred))


[CGH seul] Nested CV en cours...


/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:124

[CGH seul] Test de permutation...


/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:124

,model,nested_cv_mean,nested_cv_std,train_balanced_accuracy,test_balanced_accuracy,test_bal_acc_CI_95,test_macro_f1,train_test_gap,permutation_score,p_value,best_params
0,CGH seul,0.274,0.153,0.662,0.378,"[0.194, 0.600]",0.338,0.285,0.438,0.126,"{'clf__C': 0.1, 'clf__l1_ratio': 0.2, 'select_..."


,pred_cort,pred_dipg,pred_midl
true_cort,4,0,1
true_dipg,3,2,1
true_midl,1,2,0


## 6. Modèle 3 — Fusion précoce (sélection par bloc, puis concaténation)

On sélectionne `k_GE` features dans le bloc GE et `k_CGH` features dans CGH **séparément** (via `ColumnTransformer`), puis on concatène avant la régression. Cela évite que SelectKBest, appliqué sur le bloc concaténé brut, ne soit dominé mécaniquement par GE (15 702 features vs 1 229).


In [7]:
block_selector = ColumnTransformer([
    ("ge",  Pipeline([("var", VarianceThreshold()), ("sel", SelectKBest(f_classif))]),
            make_column_selector(pattern=r"^GE__")),
    ("cgh", Pipeline([("var", VarianceThreshold()), ("sel", SelectKBest(f_classif))]),
            make_column_selector(pattern=r"^CGH__")),
])

early_pipeline = Pipeline([
    ("blocks",   block_selector),
    ("scale",    StandardScaler()),
    ("clf",      make_logreg()),
])

early_grid = {
    "blocks__ge__sel__k":  GE_K_GRID,
    "blocks__cgh__sel__k": CGH_K_GRID,
    "clf__C":              C_GRID,
    "clf__l1_ratio":       L1_GRID,
}

early_result, early_search, _, early_test_pred = evaluate_pipeline(
    "Fusion précoce",
    early_pipeline, early_grid,
    X_concat_train, y_train, X_concat_test, y_test,
)
display(early_result.to_frame().T)
display(confusion_table(y_test, early_test_pred))


[Fusion précoce] Nested CV en cours...


/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/ruben/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:124

KeyboardInterrupt: 

## 7. Modèle 4 — Fusion tardive (Stacking, prédictions out-of-fold)

**Critique de l'ancien code** : optimiser `α` sur des `predict_proba` produites par des modèles déjà entraînés sur tout le train = data leakage partiel. Le stacking corrige ça.

`StackingClassifier` :
1. Entraîne deux pipelines bases (un sur GE, un sur CGH).
2. Calcule des prédictions **out-of-fold** (chaque échantillon prédit par un modèle qui ne l'a jamais vu).
3. Entraîne un méta-classifieur (logistique L2) sur ces prédictions OOF.

Les hyperparamètres des bases sont fixés (régularisation forte, justifiée par p ≫ n) — seul le `C` du méta-modèle est optimisé. Cela évite l'explosion combinatoire de la nested CV.


In [ ]:
# Pipelines base par bloc — sélectionnent leur bloc via ColumnTransformer puis traitent
def make_block_pipeline(pattern, k):
    return Pipeline([
        ("select_block", ColumnTransformer([
            ("blk", "passthrough", make_column_selector(pattern=pattern)),
        ])),
        ("variance", VarianceThreshold()),
        ("select",   SelectKBest(f_classif, k=k)),
        ("scale",    StandardScaler()),
        ("clf",      LogisticRegression(
            penalty="elasticnet", solver="saga",
            class_weight="balanced", max_iter=8000, random_state=SEED,
            C=0.1, l1_ratio=0.5,    # HP fixés a priori (forte régularisation)
        )),
    ])

ge_base  = make_block_pipeline(r"^GE__",  k=80)
cgh_base = make_block_pipeline(r"^CGH__", k=40)

stacking = StackingClassifier(
    estimators=[("ge", ge_base), ("cgh", cgh_base)],
    final_estimator=LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=SEED,
    ),
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED),
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1,
)

late_grid = {
    "final_estimator__C": [0.1, 1.0, 10.0],
}

late_result, late_search, _, late_test_pred = evaluate_pipeline(
    "Fusion tardive (Stacking)",
    stacking, late_grid,
    X_concat_train, y_train, X_concat_test, y_test,
    run_permutation=False,   # coûteux : stacking refait les bases à chaque permutation
)
display(late_result.to_frame().T)
display(confusion_table(y_test, late_test_pred))


## 8. Modèle 5 — Sparse PCA + LogReg

### Pourquoi sparse PCA ?

La **PCA classique** produit des composantes denses (combinaison de tous les 16 931 features) qui :
- ne sont pas interprétables biologiquement,
- mélangent signal et bruit technique (effet plateforme, variabilité d'âge),
- sont instables d'un fold à l'autre en p ≫ n.

La **sparse PCA** (Zou-Hastie-Tibshirani 2006) ajoute une pénalité L1 sur les chargements :

$$
\max_v \ \mathrm{Var}(Xv) - \alpha \|v\|_1
$$

Chaque composante devient alors une combinaison de quelques features seulement → **interprétable comme un proto-pathway**, **stable**, et permet de projeter dans un espace de faible dimension où chaque axe a un sens biologique.

### Pipeline

`VarianceThreshold → StandardScaler → SparsePCA → LogReg`. Tous les steps sont **dans** le Pipeline → refit à chaque fold de CV → pas de fuite.

### Coût computationnel

`SparsePCA` est lent (résout un problème de dictionary learning). La grille est volontairement modeste : `n_components ∈ {5, 10, 15, 20}`, `α ∈ {1.0, 2.0}`. Compter ~30-60 min selon la machine.


In [ ]:
spca_pipeline = Pipeline([
    ("variance", VarianceThreshold()),
    ("scale",    StandardScaler()),
    ("spca",     SparsePCA(random_state=SEED, max_iter=200, tol=1e-4, n_jobs=1)),
    ("clf",      make_logreg()),
])

spca_grid = {
    "spca__n_components": SPCA_N_COMPONENTS,
    "spca__alpha":        SPCA_ALPHA,
    "clf__C":             C_GRID,
    "clf__l1_ratio":      L1_GRID,
}

spca_result, spca_search, _, spca_test_pred = evaluate_pipeline(
    "Sparse PCA + LogReg",
    spca_pipeline, spca_grid,
    X_concat_train, y_train, X_concat_test, y_test,
)
display(spca_result.to_frame().T)
display(confusion_table(y_test, spca_test_pred))


## 9. Comparaison des cinq stratégies

Tous les modèles sont évalués avec **le même protocole** :
- nested CV 4-fold × 5 répétitions (boucle externe pour estimation honnête, boucle interne pour HP),
- même métrique (`balanced_accuracy`),
- même test set tenu à part,
- bootstrap CI 95% identique,
- test de permutation B=500 (sauf fusion tardive — coûteux).

C'est ce qui rend la comparaison **statistiquement défendable**.


In [ ]:
comparison = pd.DataFrame([
    ge_result, cgh_result, early_result, late_result, spca_result,
]).set_index("model")

cols_show = [
    "nested_cv_mean", "nested_cv_std",
    "train_balanced_accuracy", "test_balanced_accuracy", "test_bal_acc_CI_95",
    "test_macro_f1", "train_test_gap", "p_value",
]
comparison_display = comparison[cols_show].sort_values("nested_cv_mean", ascending=False)
display(comparison_display.round(3))


### Visualisation : distribution des scores nested CV

In [ ]:
# Boxplot des scores nested CV par modèle
nested_scores_dict = {}
for name, search, X_eval in [
    ("GE seul",                  ge_search,    X_ge_train),
    ("CGH seul",                 cgh_search,   X_cgh_train),
    ("Fusion précoce",           early_search, X_concat_train),
    ("Fusion tardive (Stack)",   late_search,  X_concat_train),
    ("Sparse PCA + LogReg",      spca_search,  X_concat_train),
]:
    # Re-faire un cross_val_score pour récupérer les scores (déjà calculés en interne)
    inner = GridSearchCV(search.estimator, search.param_grid,
                         cv=INNER_CV, scoring="balanced_accuracy",
                         refit=True, n_jobs=-1)
    nested_scores_dict[name] = cross_val_score(
        inner, X_eval, y_train,
        cv=OUTER_CV, scoring="balanced_accuracy", n_jobs=-1,
    )

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(nested_scores_dict.values(), labels=nested_scores_dict.keys(),
           patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.axhline(y=1/3, color="red", linestyle="--", alpha=0.5, label="Hasard (1/3)")
ax.set_ylabel("Balanced accuracy (nested CV)")
ax.set_title("Distribution des scores nested CV (4-fold × 5 répétitions = 20 estimations / modèle)")
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 10. Analyse de stabilité — SelectKBest sur GE

Combien de fois chaque feature GE est-elle sélectionnée par `SelectKBest` à travers les 20 folds de la CV ? Une feature présente dans 100% des folds est un signal robuste ; une feature présente dans <50% est probablement instable / due au bruit.


In [ ]:
best_k_ge = ge_search.best_params_["select__k"]

vt_ge = VarianceThreshold().fit(X_ge_train)
ge_cols_after_vt = X_ge_train.columns[vt_ge.get_support()]
X_ge_vt = vt_ge.transform(X_ge_train)

selection_counts = pd.Series(0, index=ge_cols_after_vt, name="count")
n_folds = 0

for tr_idx, _ in OUTER_CV.split(X_ge_vt, y_train):
    sel = SelectKBest(f_classif, k=min(best_k_ge, X_ge_vt.shape[1]))
    sel.fit(X_ge_vt[tr_idx], y_train.iloc[tr_idx])
    selected = ge_cols_after_vt[sel.get_support()]
    selection_counts[selected] += 1
    n_folds += 1

stability_pct = (selection_counts / n_folds * 100).sort_values(ascending=False)

# Annotation gènes (optionnelle, si fichier dispo)
def annotate(idx):
    annot_path = ANNOT_DIR / "GE_annot.csv"
    if not annot_path.exists():
        return idx.tolist()
    annot = pd.read_csv(annot_path)
    annot["row_id"] = annot["row_id"].astype(str)
    m = annot.set_index("row_id")["geneSymbol"].to_dict()
    return [f"{m.get(f, f)} ({f})" for f in idx]

top_n = 25
top = stability_pct.head(top_n)
labels = annotate(top.index)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(range(len(top)), top.values, color="steelblue")
ax.set_yticks(range(len(top)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel(f"Fréquence de sélection (% des {n_folds} folds)")
ax.set_title(f"Top {top_n} features GE — stabilité SelectKBest (k={best_k_ge})")
ax.invert_yaxis()
ax.axvline(x=80, color="red", linestyle="--", alpha=0.5, label="80%")
ax.legend()
plt.tight_layout()
plt.show()

# Résumé chiffré
n_100 = (stability_pct == 100).sum()
n_80  = (stability_pct >= 80).sum()
n_50  = (stability_pct >= 50).sum()
print(f"Features sélectionnées dans 100% des folds : {n_100}")
print(f"Features sélectionnées dans  ≥80%  des folds : {n_80}")
print(f"Features sélectionnées dans  ≥50%  des folds : {n_50}")


## 11. Analyse de stabilité — chargements Sparse PCA

Une feature qui apparaît avec un chargement non-nul dans **plusieurs composantes Sparse PCA d'un fold à l'autre** est un signal robuste. On compte, pour chaque feature, dans combien de folds elle a au moins un chargement non-nul (toutes composantes confondues).

> Note : les composantes Sparse PCA ne sont pas alignables d'un fold à l'autre (ordre + signe peuvent varier). On agrège donc au niveau "feature présente dans **au moins une** composante" plutôt que "feature présente dans la PC1".


In [ ]:
best_n_spca     = spca_search.best_params_["spca__n_components"]
best_alpha_spca = spca_search.best_params_["spca__alpha"]

vt = VarianceThreshold().fit(X_concat_train)
features_after_vt = X_concat_train.columns[vt.get_support()]
X_vt = vt.transform(X_concat_train)
scaler = StandardScaler().fit(X_vt)
X_scaled = scaler.transform(X_vt)

appearance = pd.Series(0, index=features_after_vt, name="count")
n_folds = 0

print(f"Stabilité Sparse PCA (n_components={best_n_spca}, alpha={best_alpha_spca}) sur {OUTER_CV.get_n_splits(X_scaled, y_train)} folds...")

for tr_idx, _ in OUTER_CV.split(X_scaled, y_train):
    spca = SparsePCA(
        n_components=best_n_spca, alpha=best_alpha_spca,
        random_state=SEED, max_iter=200, tol=1e-4, n_jobs=1,
    )
    spca.fit(X_scaled[tr_idx])
    nonzero_per_feature = (np.abs(spca.components_) > 1e-8).any(axis=0)
    appearance[features_after_vt[nonzero_per_feature]] += 1
    n_folds += 1

appearance_pct = (appearance / n_folds * 100).sort_values(ascending=False)

# Plot top features (avec préfixes GE__ / CGH__ pour identifier la provenance)
top = appearance_pct.head(top_n)
top_labels = annotate([f.replace("GE__", "").replace("CGH__", "") for f in top.index])
top_labels = [f"[{f.split('__')[0]}] {lbl}" for f, lbl in zip(top.index, top_labels)]

fig, ax = plt.subplots(figsize=(10, 7))
colors_block = ["steelblue" if f.startswith("GE__") else "darkorange" for f in top.index]
ax.barh(range(len(top)), top.values, color=colors_block)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top_labels, fontsize=8)
ax.set_xlabel(f"Fréquence de présence (% des {n_folds} folds)")
ax.set_title(f"Top {top_n} features — stabilité Sparse PCA\n(bleu = GE, orange = CGH)")
ax.invert_yaxis()
ax.axvline(x=80, color="red", linestyle="--", alpha=0.5, label="80%")
ax.legend()
plt.tight_layout()
plt.show()

n_100 = (appearance_pct == 100).sum()
n_80  = (appearance_pct >= 80).sum()
n_50  = (appearance_pct >= 50).sum()
print(f"Features stables (100% des folds) : {n_100}")
print(f"Features stables ( ≥80% des folds) : {n_80}")
print(f"Features stables ( ≥50% des folds) : {n_50}")

# Décomposition par bloc
ge_stable_80  = appearance_pct[(appearance_pct >= 80) & appearance_pct.index.str.startswith("GE__")]
cgh_stable_80 = appearance_pct[(appearance_pct >= 80) & appearance_pct.index.str.startswith("CGH__")]
print(f"  dont GE  : {len(ge_stable_80)}")
print(f"  dont CGH : {len(cgh_stable_80)}")


### Lecture des chargements de la solution finale

On regarde aussi, pour le modèle final (refit sur tout le train), à quoi ressemble chaque composante Sparse PCA — combien de features non-nulles, et lesquelles dominent.


In [ ]:
# Récupérer la Sparse PCA du modèle final
final_spca = spca_search.best_estimator_.named_steps["spca"]

# Reconstruire les noms de features post-VarianceThreshold
final_vt = spca_search.best_estimator_.named_steps["variance"]
features_final = X_concat_train.columns[final_vt.get_support()]

components = pd.DataFrame(
    final_spca.components_,
    index=[f"SPC{i+1}" for i in range(final_spca.n_components_)],
    columns=features_final,
)

# Sparsité de chaque composante
sparsity = (components.abs() > 1e-8).sum(axis=1)
print("Nombre de features non-nulles par composante :")
print(sparsity.to_string())

# Top features par composante
print("\nTop 5 features par composante (par |loading|) :")
for pc in components.index:
    top5 = components.loc[pc].abs().sort_values(ascending=False).head(5)
    block_summary = sum(1 for f in top5.index if f.startswith("GE__"))
    print(f"\n  {pc}  (top 5, dont {block_summary}/5 GE) :")
    for f, w in top5.items():
        sign = "+" if components.loc[pc, f] > 0 else "-"
        print(f"    {sign} {f}  (loading = {components.loc[pc, f]:+.3f})")


## 12. Conclusion

Les principaux apports de cette version :

1. **Évaluation honnête** : la nested CV donne une estimation non biaisée pour les cinq stratégies, comparables entre elles.
2. **Pas de fuite par design** : grilles fixées a priori, tout le pré-traitement dans le Pipeline.
3. **Sparse PCA** : composantes interprétables (chargements creux), gènes stables identifiables, vs PCA classique qui mélange tout.
4. **Stacking** pour la fusion tardive : pas de leakage sur α (méta-modèle entraîné sur prédictions out-of-fold).
5. **Stabilité analysée des deux côtés** : SelectKBest sur GE *et* chargements Sparse PCA — les deux donnent des candidats biologiques à valider.

### Points à creuser

- **MOFA / MOFA+** pour une décomposition explicite GE-spécifique vs CGH-spécifique vs partagée.
- **DIABLO** (`mixOmics` en R) pour une fusion supervisée bloc-aware, qui devrait dominer la fusion tardive ici.
- **Cohorte externe** (OpenPBTA, CBTTC) pour valider la généralisation au-delà de l'IGR.
- **Enrichissement pathway** sur les features stables (GSEA / GO-BP) pour interprétation biologique.
